# Module 11 · 1：從資料到模型的地圖

> **Module 11 定位｜前處理之後，能訓練什麼模型？**
> M09 把文字/圖像/聲音/影片變成了規整張量與資料集。本模組回答下一個問題：
> **「資料整理好之後，可以訓練/微調什麼模型、做什麼任務？」**
> 本節給出一張**總地圖**，把資料結構與下游模型對應起來。

## 學習目標
- 建立「**資料模態 × 資料結構 → 模型家族 → 任務**」的對應地圖。
- 認識 2026 的訓練技術棧與**參數高效微調 (LoRA/QLoRA)**。
- 理解三種訓練範式：從零訓練 / 全參數微調 / 參數高效微調，以及如何取捨。

<!-- concept-image:01_data_to_model_map_fig1 -->
<div align="center">
<img src="concept_images/01_data_to_model_map_fig1_modality_map_20260611_224233.png" alt="資料模態對應地圖" width="640" style="max-width:100%;">
<br><sub>圖 1 · 資料模態對應地圖</sub>
</div>

<!-- concept-image:01_data_to_model_map_fig2 -->
<div align="center">
<img src="concept_images/01_data_to_model_map_fig2_training_paradigms_20260611_224309.png" alt="三種訓練範式對比" width="640" style="max-width:100%;">
<br><sub>圖 2 · 三種訓練範式對比</sub>
</div>

## 1. 總地圖：資料結構 → 可訓練的模型

| 模態 | 前處理後的資料結構 | 可訓練/微調的模型 | 典型任務 |
|:--|:--|:--|:--|
| 文字 | `input_ids (B,L)` | BERT/DistilBERT | 文本分類、NER |
| 文字 | chat/instruction JSONL | LLM（Llama/Qwen）+ LoRA | 對話、指令遵循 |
| 文字 | 句向量 `(N,D)` | 句向量模型 / 檢索器 | 語意檢索、RAG |
| 圖像 | `(N,C,H,W)` | ViT / CLIP / Resnet | 影像分類、檢索 |
| 圖像+文字 | `{image, text}` 配對 | CLIP | 跨模態檢索、zero-shot |
| 聲音 | log-mel `(N,80,T)` | Whisper | 語音辨識 (ASR) |
| 聲音 | 波形 `(N,samples)` | wav2vec2/HuBERT | 語音分類、表示學習 |
| 影片 | `(N,T,C,H,W)` | VideoMAE/ViViT | 動作辨識、時序定位 |
| 圖+文對話 | VLM messages | VLM（LLaVA/Qwen-VL） | 看圖問答 |

**核心觀念**：**資料的形狀與格式，決定了你能接哪一類模型**。
這正是 M09 一直強調「資料結構設計」的原因。

In [1]:
# 一個離線的「決策小幫手」：輸入模態與任務，回傳建議模型與資料格式
DECISION = {
    ("text", "classification"): ("DistilBERT/BERT 微調", "input_ids (B,L) + 標籤 (B,)"),
    ("text", "chat"):           ("LLM + LoRA 指令微調", "chat JSONL messages"),
    ("text", "retrieval"):      ("sentence-transformers", "句向量 (N,D) + 餘弦相似度"),
    ("image", "classification"):("ViT 微調 / CLIP zero-shot", "(N,C,H,W) + 標籤 (B,)"),
    ("audio", "asr"):           ("Whisper 微調", "log-mel (N,80,T) + 文字標籤"),
    ("audio", "classification"):("wav2vec2 微調", "波形 (N,samples) + 標籤 (B,)"),
    ("video", "action"):        ("VideoMAE 微調", "(N,T,C,H,W) + 標籤 (B,)"),
}
def recommend(modality, task):
    model, fmt = DECISION.get((modality, task), ("（請參考總地圖）", "—"))
    print(f"模態={modality}, 任務={task}\n  建議模型: {model}\n  資料格式: {fmt}")

recommend("text", "chat")
recommend("audio", "asr")
recommend("video", "action")

模態=text, 任務=chat
  建議模型: LLM + LoRA 指令微調
  資料格式: chat JSONL messages
模態=audio, 任務=asr
  建議模型: Whisper 微調
  資料格式: log-mel (N,80,T) + 文字標籤
模態=video, 任務=action
  建議模型: VideoMAE 微調
  資料格式: (N,T,C,H,W) + 標籤 (B,)


## 2. 2026 訓練技術棧

| 套件 | 角色 |
|:--|:--|
| `torch` | 張量與自動微分 |
| `transformers` | 模型 + `Trainer` 訓練迴圈 |
| `datasets` | 資料載入/前處理/串流 |
| `peft` | 參數高效微調（LoRA/QLoRA） |
| `trl` | SFT、偏好對齊（DPO） |
| `accelerate` | 多 GPU / 混合精度 |
| `evaluate` | 指標計算 |

安裝：`uv sync --extra multimodal --extra train`

<!-- concept-image:01_data_to_model_map_fig3 -->
<div align="center">
<img src="concept_images/01_data_to_model_map_fig3_tech_stack_20260611_224346.png" alt="2026 訓練技術棧生態" width="640" style="max-width:100%;">
<br><sub>圖 3 · 2026 訓練技術棧生態</sub>
</div>

## 3. 三種訓練範式（如何取捨）

| 範式 | 更新哪些參數 | 算力/資料需求 | 何時用 |
|:--|:--|:--|:--|
| 從零預訓練 | 全部（隨機起始） | 極高（海量資料+大量 GPU） | 幾乎不自己做 |
| 全參數微調 | 全部（從預訓練起始） | 高 | 資料多、要極致效果 |
| **參數高效微調 (LoRA/QLoRA)** | 只加一小撮新參數 | **低（單卡甚至可跑）** | **2026 最常見的客製化方式** |

**LoRA 直覺**：凍結原模型，只在每層插入兩個小矩陣去學「差異」，
可訓練參數常 < 1%，卻能逼近全參數微調效果——這讓「用自己的資料微調大模型」變得人人可及。

## 小結
- 資料結構 → 模型家族 → 任務，是貫穿 M09→M11 的主軸。
- 2026 客製化大模型的主流是**參數高效微調**。
- 接下來各節依模態示範「資料 → 模型」的最小流程。